In [ ]:
# given a g, r, or i magnitude, abs magnitude, what is full abs magnitude of the TDE

In [ ]:
# given an abs magnitude at peak, redshift, and days since peak, what is the apparent mag of the TDE, and the g r i mags
                                                                                                                                                                                           
from thor.utils.rates_sims import tde_apparent_mags

mags = tde_apparent_mags(M_abs=-22.01, z=2, days_since_peak=60)

In [ ]:
# how much brighter magnitude will a flat spectrum be if at z=1 instead of z=2

from astropy.cosmology import Planck18
import numpy as np

d1 = Planck18.luminosity_distance(1.0)
d2 = Planck18.luminosity_distance(2.0)

delta_m = 5 * np.log10(d2 / d1)
print(f"d_L(z=1) = {d1:.1f}")
print(f"d_L(z=2) = {d2:.1f}")
print(f"z=1 is {delta_m:.2f} mag brighter than z=2")

In [ ]:
# given an SED, what is the magnitude in x filter

import numpy as np
from astropy import units as u
from synphot import SourceSpectrum, SpectralElement, Observation
from synphot.models import BlackBodyNorm1D, Empirical1D


# Valid filter names for SpectralElement.from_filter():
# 'bessel_j', 'bessel_h', 'bessel_k'
# 'cousins_r', 'cousins_i'
# 'johnson_u', 'johnson_b', 'johnson_v', 'johnson_r', 'johnson_i', 'johnson_j', 'johnson_k'

def blackbody_magnitude(T_K, filter_band, system='vegamag'):
    """
    Compute synthetic magnitude of a 1-R_sun blackbody at 10 pc through a filter.

    Parameters
    ----------
    T_K : float
        Blackbody temperature [K]
    filter_band : str or SpectralElement
        e.g. 'bessel_k', 'johnson_v', 'johnson_b'
    system : str
        'vegamag' or 'abmag'
    """
    # Load filter first to know wavelength range
    if isinstance(filter_band, str):
        filt = SpectralElement.from_filter(filter_band)
    else:
        filt = filter_band

    # Build source on a grid wide enough to cover the filter
    # (BlackBodyNorm1D defaults to ~1-10000 AA which misses NIR filters like K-band)
    w_min = max(500,  filt.waveset.min().value * 0.5)
    w_max = min(1e6,  filt.waveset.max().value * 2.0)
    wave = np.linspace(w_min, w_max, 100_000) * u.AA

    bb = SourceSpectrum(BlackBodyNorm1D, temperature=T_K * u.K)
    src = SourceSpectrum(Empirical1D, points=wave, lookup_table=bb(wave))

    obs = Observation(src, filt)
    return obs.effstim(system)


# ── Pure-numpy version (works with any SED array) ─────────────────────────────

def synthetic_mag_numpy(wave_AA, f_lam, trans, system='AB'):
    """
    Compute synthetic magnitude from arrays.

    Parameters
    ----------
    wave_AA : array
        Wavelength grid [Angstroms], must match f_lam and trans
    f_lam : array
        Source flux density f_lambda [erg/s/cm²/Å]
    trans : array
        Filter transmission T(λ), dimensionless [0–1], photon-counting
    system : str
        'AB' or 'ST'
    """
    c_AA = 2.998e18                            # speed of light in Å/s
    f_nu = f_lam * wave_AA**2 / c_AA          # f_lambda -> f_nu [erg/s/cm²/Hz]

    # Photon-counting: <f_nu> = ∫ f_nu * T(λ) λ dλ / ∫ T(λ) λ dλ
    num = np.trapz(f_nu * trans * wave_AA, wave_AA)
    den = np.trapz(        trans * wave_AA, wave_AA)
    f_nu_eff = num / den

    if system == 'AB':
        return -2.5 * np.log10(f_nu_eff) - 48.6
    elif system == 'ST':
        f_lam_eff = np.trapz(f_lam * trans, wave_AA) / np.trapz(trans, wave_AA)
        return -2.5 * np.log10(f_lam_eff) - 21.1
    else:
        raise ValueError(f"Unknown system: {system}")


# ── Example: 30,000 K blackbody in Bessel K ───────────────────────────────────

mag_vega = blackbody_magnitude(30_000, 'bessel_k', system='vegamag')
mag_ab   = blackbody_magnitude(30_000, 'bessel_k', system='abmag')
print(f"Bessel K (Vega): {mag_vega:.3f}")
print(f"Bessel K (AB):   {mag_ab:.3f}")

# --- numpy approach (tophat K-band approximation) ---
T = 30_000
wave = np.linspace(1e3, 5e4, 50_000)   # Å, 0.1–5 µm

h, c, k = 6.626e-27, 2.998e10, 1.381e-16
wave_cm = wave * 1e-8
f_lam = (2*h*c**2 / wave_cm**5) / (np.exp(h*c / (wave_cm*k*T)) - 1) * 1e-8  # /Å

# Tophat filter: K-band  2.0–2.4 µm
trans = np.where((wave >= 2.0e4) & (wave <= 2.4e4), 1.0, 0.0)

mag_np = synthetic_mag_numpy(wave, f_lam, trans, system='AB')
print(f"K-band tophat (AB, numpy):    {mag_np:.3f}  (un-normalized, relative only)")